# 07 — Affective Dialog: Tone & Style Adaptation

## What Is Affective Dialog?

**Affective dialog** refers to conversation systems that sense the user's
emotional state and adapt their tone, vocabulary, and energy accordingly.
Instead of a one-size-fits-all assistant, you get a model that:

- Matches excitement when the user is excited
- Slows down and empathizes when the user is frustrated
- Stays neutral and educational when the user wants information

### How Does the Model Adapt?

The Gemini Live API provides two levers:

1. **System prompt** — defines the persona and emotional rules  
2. **Voice selection** — each voice has a distinct personality that reinforces tone

### Use Cases
- **Mental health / wellness apps** — empathetic, calm responses to distressed users  
- **Education platforms** — energetic encouragement for children, formal explanations for adults  
- **Customer service** — de-escalation for angry callers, enthusiasm for positive interactions  
- **Entertainment** — character-driven NPCs with distinct personalities

---

**Notebook structure**
1. Setup  
2. Demo 1 — Voice style comparison (4 tones, same question)  
3. Demo 2 — Emotion-responsive system  
4. Demo 3 — Voice persona comparison  
5. Available voices reference table  
6. Key takeaways

## Setup

In [ ]:
!pip install -q google-genai numpy

In [ ]:
import nest_asyncio; nest_asyncio.patch()

import asyncio
import os
import json
import numpy as np
import IPython.display as ipd
from google import genai
from google.genai import types

from dotenv import load_dotenv
load_dotenv()  # loads GEMINI_API_KEY from .env

API_KEY = os.environ.get("GEMINI_API_KEY", "")
MODEL   = "gemini-3.1-flash-live-preview"

client = genai.Client(api_key=API_KEY)
print(f"Client ready. Model: {MODEL}")

In [ ]:
# ── audio helpers ─────────────────────────────────────────────────────────────

def play_pcm(raw_bytes: bytes, rate: int = 24000) -> ipd.Audio:
    """Wrap raw PCM bytes in an IPython Audio widget."""
    arr = np.frombuffer(raw_bytes, dtype=np.int16).astype(np.float32) / 32768.0
    return ipd.Audio(arr, rate=rate, autoplay=False)


async def run_turn(
    user_text: str,
    system_prompt: str,
    voice_name: str = "Aoede",
    label: str = "",
) -> dict:
    """
    Open a session, send user_text, collect audio + transcript.
    Returns {'audio': bytes, 'transcript': str}.
    """
    config = types.LiveConnectConfig(
        response_modalities=["AUDIO"],
        output_audio_transcription=types.AudioTranscriptionConfig(),
        system_instruction=system_prompt,
        speech_config=types.SpeechConfig(
            voice_config=types.VoiceConfig(
                prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name=voice_name)
            )
        ),
    )

    audio_chunks = []
    transcript_parts = []

    async with client.aio.live.connect(model=MODEL, config=config) as session:
        await session.send_realtime_input(text=user_text)

        async for resp in session.receive():
            if resp.data:
                audio_chunks.append(resp.data)

            sc = resp.server_content
            if sc:
                if sc.output_transcription and sc.output_transcription.text:
                    transcript_parts.append(sc.output_transcription.text)
                if sc.model_turn:
                    for part in sc.model_turn.parts:
                        if hasattr(part, "text") and part.text:
                            transcript_parts.append(part.text)
                if sc.turn_complete:
                    break
                if sc.interrupted:
                    break
            if resp.go_away:
                break

    result = {
        "audio": b"".join(audio_chunks),
        "transcript": "".join(transcript_parts),
    }
    if label:
        print(f"[{label}] done — {len(result['audio']):,} bytes audio")
    return result


print("Helpers defined.")

---
## Demo 1 — Voice Style Comparison (4 Tones, Same Question)

We send **the same question** to four differently-prompted personas and compare
how each one answers. The content is nearly identical; the **delivery** differs.

This demonstrates how the system prompt alone controls emotional register.

In [ ]:
# ── Define 4 tonal personas ────────────────────────────────────────────────

PERSONAS = {
    "Professional & Formal": (
        "You are a formal corporate assistant. Use precise, professional language. "
        "Avoid contractions and colloquialisms. Maintain a composed, authoritative tone.",
        "Kore",   # firm, professional voice
    ),
    "Casual & Friendly": (
        "You are a casual, friendly assistant. Use contractions, simple words, and "
        "feel free to use informal phrases. Talk like a helpful friend.",
        "Puck",   # upbeat, playful voice
    ),
    "Excited & Enthusiastic": (
        "You are an extremely enthusiastic assistant. Show genuine excitement about "
        "everything. Use exclamation points, emphasize interesting facts, be high-energy.",
        "Fenrir", # excitable voice
    ),
    "Calm & Empathetic": (
        "You are a calm, empathetic assistant. Speak slowly and deliberately. "
        "Acknowledge the user's perspective before answering. Be warm and reassuring.",
        "Aoede",  # warm, breezy voice
    ),
}

SAME_QUESTION = "Can you explain how photosynthesis works?"

print(f"Question: '{SAME_QUESTION}'")
print(f"Testing {len(PERSONAS)} personas...")

In [ ]:
# Run all 4 personas sequentially (each opens its own session)

async def run_all_personas():
    results = {}
    for name, (prompt, voice) in PERSONAS.items():
        print(f"\nRunning persona: {name!r} (voice={voice})")
        r = await run_turn(
            user_text=SAME_QUESTION,
            system_prompt=prompt,
            voice_name=voice,
            label=name,
        )
        results[name] = r
    return results


persona_results = asyncio.run(run_all_personas())

print("\n" + "=" * 60)
print("TRANSCRIPT COMPARISON")
print("=" * 60)
for name, r in persona_results.items():
    print(f"\n--- {name} ---")
    print(r["transcript"] or "(no transcript — audio only)")

In [ ]:
# Play audio for each persona
for name, r in persona_results.items():
    if r["audio"]:
        print(f"\n--- Audio: {name} ---")
        display(play_pcm(r["audio"]))
    else:
        print(f"No audio for {name}")

---
## Demo 2 — Emotion-Responsive System

A single system prompt can instruct the model to **detect** the emotional
register of the user's message and adapt its reply accordingly.

We send three messages with very different emotional contexts and observe
how the model's tone shifts:

| User message | Expected model tone |
|-------------|--------------------|
| "I just got promoted!" | Excited, celebratory |
| "Been debugging for 6 hours..." | Empathetic, supportive |
| "Explain quantum computing" | Neutral, educational |

In [ ]:
EMOTION_ADAPTIVE_PROMPT = """\
You are an emotionally intelligent assistant. Carefully read the emotional
context of the user's message and adapt your tone accordingly:

- If they express excitement or joy: match their energy, be celebratory and warm.
- If they express frustration or exhaustion: slow down, be empathetic, validate
  their feelings before offering help.
- If they ask a neutral information question: be clear, calm, and educational.

Always keep your reply under 3 sentences. Tone first, content second.
"""

# Three messages with distinct emotional registers
EMOTIONAL_MESSAGES = [
    (
        "I'm SO EXCITED! I just got a promotion to Senior Engineer!",
        "Expected: excited / celebratory",
    ),
    (
        "I've been trying to fix this stupid bug for 6 hours and nothing works...",
        "Expected: empathetic / supportive",
    ),
    (
        "Can you explain quantum computing in simple terms?",
        "Expected: neutral / educational",
    ),
]


async def demo_emotion_responsive():
    results = []
    for msg, expected in EMOTIONAL_MESSAGES:
        print(f"\nUser: {msg!r}")
        print(f"  {expected}")
        r = await run_turn(
            user_text=msg,
            system_prompt=EMOTION_ADAPTIVE_PROMPT,
            voice_name="Aoede",
            label="EmoAdaptive",
        )
        results.append((msg, expected, r))
    return results


emotion_results = asyncio.run(demo_emotion_responsive())

print("\n" + "=" * 60)
print("EMOTION-ADAPTIVE RESPONSES")
print("=" * 60)
for msg, expected, r in emotion_results:
    print(f"\n[{expected}]")
    print(f"User: {msg}")
    print(f"Model: {r['transcript'] or '(audio only)'}")

In [ ]:
# Play audio for each emotional response
for i, (msg, expected, r) in enumerate(emotion_results, 1):
    short_msg = msg[:50] + "..." if len(msg) > 50 else msg
    print(f"\n--- Response {i}: {expected} ---")
    print(f"    User: {short_msg}")
    if r["audio"]:
        display(play_pcm(r["audio"]))
    else:
        print("    (no audio returned)")

---
## Demo 3 — Voice Persona Comparison

The Gemini Live API ships with several **pre-built voices**, each with a
distinct acoustic personality. Even with the same system prompt,
different voices create very different listener experiences.

We'll send the same question with 5 different voices and collect transcripts
to compare. The actual audio files demonstrate the personality differences
most clearly — play them after the cell runs.

### Available Voices

| Voice | Gender | Personality |
|-------|--------|-------------|
| Aoede | Female | Warm, breezy — good for empathetic use cases |
| Kore | Female | Firm, professional — good for corporate assistants |
| Charon | Male | Deep, informative — good for narration / education |
| Fenrir | Male | Excitable — good for gaming, entertainment |
| Puck | Male | Upbeat, playful — good for consumer apps |
| Zephyr | Female | Light, airy — good for wellness / meditation |

In [ ]:
# Voices to compare
VOICE_LIST = [
    ("Aoede",  "Female", "Warm, breezy"),
    ("Kore",   "Female", "Firm, professional"),
    ("Charon", "Male",   "Deep, informative"),
    ("Fenrir", "Male",   "Excitable"),
    ("Puck",   "Male",   "Upbeat, playful"),
]

VOICE_QUESTION = "What is the most fascinating thing about the ocean?"

NEUTRAL_PROMPT = (
    "You are a knowledgeable assistant. Answer in 2 sentences, clearly and engagingly."
)


async def demo_voice_comparison():
    results = {}
    for voice_name, gender, personality in VOICE_LIST:
        print(f"\nVoice: {voice_name} ({gender}, {personality})")
        r = await run_turn(
            user_text=VOICE_QUESTION,
            system_prompt=NEUTRAL_PROMPT,
            voice_name=voice_name,
            label=voice_name,
        )
        results[voice_name] = r
    return results


voice_results = asyncio.run(demo_voice_comparison())

print("\n" + "=" * 60)
print(f"Question: '{VOICE_QUESTION}'")
print("=" * 60)
for voice_name, r in voice_results.items():
    print(f"\n[{voice_name}]: {r['transcript'] or '(audio only)'}")

In [ ]:
# Display audio widgets for all voices side by side for easy comparison
for voice_name, r in voice_results.items():
    vinfo = next((g + " — " + p for v, g, p in VOICE_LIST if v == voice_name), "")
    print(f"\n--- {voice_name} ({vinfo}) ---")
    if r["audio"]:
        display(play_pcm(r["audio"]))
    else:
        print("  (no audio)")

---
## Complete Available Voices Reference

| Voice | Gender | Personality | Best Use Case |
|-------|--------|-------------|---------------|
| **Aoede** | Female | Warm, breezy | Customer service, empathetic assistants |
| **Kore** | Female | Firm, professional | Corporate, finance, legal assistants |
| **Charon** | Male | Deep, informative | Education, narration, documentation |
| **Fenrir** | Male | Excitable | Gaming, sports, entertainment |
| **Puck** | Male | Upbeat, playful | Consumer apps, social, casual assistants |
| **Zephyr** | Female | Light, airy | Wellness, meditation, mindfulness apps |

### How to Set the Voice

```python
speech_config = types.SpeechConfig(
    voice_config=types.VoiceConfig(
        prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name="Kore")
    )
)

config = types.LiveConnectConfig(
    response_modalities=["AUDIO"],
    speech_config=speech_config,
    ...
)
```

### Matching Voice to Use Case

```
Use Case                 Recommended Voice   Why
─────────────────────    ─────────────────   ───────────────────────────────────
HR onboarding chatbot    Aoede              Warm, approachable
Legal / compliance IVR   Kore               Authoritative, precise
Audiobook narrator       Charon             Rich, deep, engaging
Kids learning app        Fenrir / Puck      High energy, fun
Meditation guide         Zephyr             Soft, calming
```

---
## Bonus: Dynamic Persona Switching

In some applications you may want to switch personas **mid-conversation**
based on detected user sentiment. The pattern below shows how to do this
by opening a new session with a different system prompt when a sentiment
threshold is crossed.

> **Note**: Voice cannot be changed within an active session. You must
> open a new session to change the voice.

In [ ]:
# Simplified sentiment detection (keyword-based for demo purposes)
# In production, you'd use a classifier or the model itself to detect sentiment.

def detect_sentiment(text: str) -> str:
    """Very simple keyword-based sentiment tagger."""
    text_lower = text.lower()
    if any(w in text_lower for w in ["frustrated", "stuck", "broken", "hours", "still not"]):
        return "frustrated"
    if any(w in text_lower for w in ["excited", "amazing", "awesome", "love", "great"]):
        return "excited"
    return "neutral"


PERSONA_MAP = {
    "frustrated": (
        "Calm, empathetic supporter. Start with 'That sounds really tough.' Be supportive.",
        "Aoede",
    ),
    "excited": (
        "Enthusiastic cheerleader. Match the user's excitement. Use exclamation points!",
        "Fenrir",
    ),
    "neutral": (
        "Clear, helpful assistant. Answer concisely in one or two sentences.",
        "Kore",
    ),
}

# Test messages
TEST_MESSAGES = [
    "I've been stuck on this Docker issue for hours and still not working!",
    "This is so amazing, I just deployed my first ML model!",
    "What is gradient descent?",
]


async def demo_dynamic_persona():
    for msg in TEST_MESSAGES:
        sentiment = detect_sentiment(msg)
        system_prompt, voice = PERSONA_MAP[sentiment]

        print(f"\nUser: {msg}")
        print(f"  Detected sentiment: {sentiment!r}  →  voice={voice}")

        r = await run_turn(
            user_text=msg,
            system_prompt=system_prompt,
            voice_name=voice,
            label=sentiment,
        )
        print(f"  Model: {r['transcript'] or '(audio only)'}")
        if r["audio"]:
            display(play_pcm(r["audio"]))


asyncio.run(demo_dynamic_persona())

---
## Key Takeaways

| Concept | Key Point |
|---------|----------|
| **System prompt = tone controller** | The fastest way to change emotional register is a well-crafted system prompt |
| **Voice = acoustic personality** | Choose voice to match the use case; voice reinforces prompt tone |
| **Emotion-responsive** | A single prompt can instruct the model to adapt based on user cues |
| **Dynamic switching** | Detect sentiment outside the model → open a new session with the right voice |
| **Cannot change voice mid-session** | Voice is set at connection time; start a new session to switch |
| **Transcript = tone evidence** | `output_audio_transcription` gives you text to analyze or log |

### Design Principles for Affective Dialog

1. **Define emotional states explicitly** in the system prompt — don't rely on the model to infer
2. **Pair voice with persona** — a professional prompt with an excitable voice creates dissonance
3. **Test all emotional paths** — make sure frustrated-user handling doesn't sound dismissive
4. **Log transcripts** — use `output_audio_transcription` to monitor tone drift in production

### Next Steps
- **Notebook 08** — Google Search grounding and session management